In [0]:
%pip install python-dotenv --quiet
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv

load_dotenv("/Workspace/Users/ik.kukoo@gmail.com/.env", override=True)

STORAGE_ACCOUNT = "internshipdatalake"
CONTAINER       = "raw"

adls_options = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

PATH_ESTOQUE = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/batch-data/food_estoque_lojas.csv"

print("✅ Config OK →", PATH_ESTOQUE)

In [0]:
# Verificando o conteúdo do arquivo food_estoque_lojas.csv
df = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PATH_ESTOQUE)
)

print(f"Linhas: {df.count()} | Colunas: {len(df.columns)}")
df.printSchema()

In [0]:
# Amostra dos dados
df.show(10, truncate=False)

In [0]:
# Estatísticas descritivas
df.describe().show(truncate=False)

In [0]:
# Análise de nulos
from pyspark.sql.functions import col, sum as spark_sum

nulls = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

print("🔍 Contagem de nulos por coluna:")
nulls.show(truncate=False)

In [0]:
# Distribuição de estoque por loja
from pyspark.sql.functions import avg, min as spark_min, max as spark_max, count

df.groupBy("id_loja").agg(
    count("sku").alias("total_skus"),
    avg("quantidade_disponivel").alias("media_qtd"),
    spark_min("quantidade_disponivel").alias("min_qtd"),
    spark_max("quantidade_disponivel").alias("max_qtd"),
    avg("estoque_minimo").alias("media_estoque_minimo")
).orderBy("id_loja").show(20, truncate=False)

In [0]:
# Itens abaixo do estoque mínimo (alerta crítico)
from pyspark.sql.functions import when

df_critico = df.filter(col("quantidade_disponivel") < col("estoque_minimo"))

print(f"⚠️ Itens abaixo do estoque mínimo: {df_critico.count()} de {df.count()} ({df_critico.count()/df.count()*100:.1f}%)")
df_critico.show(10, truncate=False)

In [0]:
# Evolução temporal (snapshots)
df.groupBy("dt_snapshot").count().orderBy("dt_snapshot").show(30, truncate=False)